In [5]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os
import sys

import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))




2.5.1+cu121
True
NVIDIA GeForce RTX 3060 Ti


In [6]:
import os

path_a = "../data/raw/set_A/"
files = os.listdir(path_a)
print(files)
print(os.path.getsize(os.path.join(path_a, files[0])) / 1024, "KB")


FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: '../data/raw/set_A/'

In [7]:
# Configuración
path_a = "../data/raw/set_A/"
path_e = "../data/raw/set_E/"


def load_first_file(folder):
    files = [f for f in os.listdir(folder) if f.endswith('.txt')]
    file_path = os.path.join(folder, files[0])
    return np.genfromtxt(file_path, dtype=float, delimiter=None)
sig_normal = load_first_file(path_a)
sig_seizure = load_first_file(path_e)

plt.figure(figsize=(15, 6))
plt.subplot(2, 1, 1)
plt.plot(sig_normal, color='blue')
plt.title("Señal Normal (Set A)")
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(sig_seizure, color='red')
plt.title("Señal de Crisis (Set E)")
plt.grid(True)

plt.tight_layout()
plt.show()



FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: '../data/raw/set_A/'

In [8]:
# --- TU EJERCICIO DE CÓDIGO ---

# 1. Definimos la frecuencia de muestreo dada por el dataset
fs = 173.61

# 2. Creamos un array que represente el tiempo en segundos para cada muestra.
# Ayuda: debe ir de 0 a (TotalMuestras / fs) en pasos de (1 / fs)
# Puedes usar np.linspace o np.arange.
n_samples = len(sig_normal)
time_seconds = np.linspace(0, n_samples / fs, n_samples)

# 3. Calculamos cuántas muestras equivalen a 5 segundos
# Ayuda: fs (muestras/seg) * 5 (seg)
samples_5s = int(fs * 5)

# 4. Visualización de los primeros 5 segundos
plt.figure(figsize=(15, 6))

plt.subplot(2, 1, 1)
# IMPORTANTE: Ahora pasamos 'time_seconds' como eje X y recortamos la señal hasta 'samples_5s'
plt.plot(time_seconds[:samples_5s], sig_normal[:samples_5s], color='blue')
plt.title("Primeros 5 Segundos - Señal Normal")
plt.xlabel("Tiempo (segundos)") # ¡Ahora sí!
plt.ylabel("Amplitud ($\mu$V)") # Bonn suele estar en microvoltios
plt.grid(True)

plt.subplot(2, 1, 2)
# Hacemos lo mismo para la señal de crisis
plt.plot(time_seconds[:samples_5s], sig_seizure[:samples_5s], color='red')
plt.title("Primeros 5 Segundos - Señal de Crisis")
plt.xlabel("Tiempo (segundos)")
plt.ylabel("Amplitud ($\mu$V)")
plt.grid(True)

plt.tight_layout()
plt.show()

NameError: name 'sig_normal' is not defined

In [14]:
import os

def build_giant_matrix(folder_path, label, window_size=174):
    """
    Lee todos los .txt de una carpeta, los trocea en ventanas 
    y les asigna una etiqueta.
    """
    all_windows = []
    all_labels = []
    
    # Listamos todos los archivos .txt de la carpeta
    files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    print(f"Procesando {len(files)} archivos en {folder_path}...")
    
    for f in files:
        # Cargar la señal completa (4097 puntos)
        signal = np.loadtxt(os.path.join(folder_path, f))
        
        # Troceamos la señal en ventanas de ~1 segundo (174 puntos)
        # Sin solapamiento por ahora para hacerlo simple
        for i in range(0, len(signal) - window_size, window_size):
            window = signal[i : i + window_size]
            
            # NORMALIZACIÓN: Paso vital en EEG. 
            # Restamos media y dividimos por desviación estándar.
            window = (window - np.mean(window)) / (np.std(window) + 1e-8)
            
            all_windows.append(window)
            all_labels.append(label)
            
    return np.array(all_windows), np.array(all_labels)

# --- EJECUCIÓN ---

# Definimos las rutas (ajusta según tus carpetas)
path_a = "../data/BONN/raw/set_A/"
path_e = "../data/BONN/raw/set_E/"

# Procesamos ambos sets
X_a, y_a = build_giant_matrix(path_a, label=0) # 0 = Sano
X_e, y_e = build_giant_matrix(path_e, label=1) # 1 = Crisis

# Concatenamos todo en una sola matriz gigante
X = np.concatenate([X_a, X_e], axis=0)
y = np.concatenate([y_a, y_e], axis=0)

print("-" * 30)
print(f"Forma final de X: {X.shape}") # Debería ser algo como (4600, 174)
print(f"Forma final de y: {y.shape}")

Procesando 100 archivos en ../data/BONN/raw/set_A/...
Procesando 100 archivos en ../data/BONN/raw/set_E/...
------------------------------
Forma final de X: (4600, 174)
Forma final de y: (4600,)


In [15]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class BonnDataset(Dataset):
    def __init__(self, X, y):
        # Convertimos NumPy a Tensores de PyTorch
        # X necesita una dimensión extra para el 'canal' (1 canal porque es EEG monocanal)
        # Forma original: (N, 174) -> Forma para CNN 1D: (N, 1, 174)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- CREACIÓN DEL DATASET Y PARTICIÓN ---

dataset_completo = BonnDataset(X, y)

total_samples = len(dataset_completo)
train_size = int(0.7 * total_samples)
val_size = int(0.15 * total_samples)
test_size = total_samples - train_size - val_size

# Hacemos la división
train_dataset, val_dataset, test_dataset = random_split(
    dataset_completo, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42) # Semilla para que siempre sea igual
)

# Creamos los 3 DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Muestras de entrenamiento: {len(train_dataset)}")
print(f"Muestras de validación: {len(val_dataset)}")
print(f"Muestras de test: {len(test_dataset)}")

Muestras de entrenamiento: 3220
Muestras de validación: 690
Muestras de test: 690


In [16]:
import torch
import torch.nn as nn

class SimpleCNNTransformer(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleCNNTransformer, self).__init__()
        
        # --- BLOQUE CNN ---
        # Extraemos patrones locales. Input: (Batch, 1, 174)
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=1, padding=3), # Mantiene longitud
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2), # Reduce a (Batch, 16, 87)
            
            nn.Conv1d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(32) # Forzamos una longitud fija para el Transformer: (Batch, 32, 32)
        )
        
        # --- BLOQUE TRANSFORMER ---
        # El Transformer espera: (Secuencia, Batch, Feature)
        d_model = 32 # Dimensión de las características
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=4,         # 4 cabezas de atención
            dim_feedforward=64, 
            dropout=0.1,
            batch_first=True # Para que acepte (Batch, Seq, Feature)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        
        # --- CLASIFICADOR ---
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)              # Out: (Batch, 32, 32)
        x = x.permute(0, 2, 1)       # Transformer quiere (Batch, Seq, Feature) -> (Batch, 32, 32)
        x = self.transformer(x)
        x = self.classifier(x)
        return x

# Instanciar el modelo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNNTransformer().to(device)
print(f"Modelo listo en: {device}")

Modelo listo en: cuda


In [17]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100 * correct / len(train_dataset)

# Prueba 5 épocas rápidas
for epoch in range(1, 6):
    loss, acc = train_one_epoch()
    print(f"Época {epoch} -> Loss: {loss:.4f} | Acc: {acc:.2f}%")

Época 1 -> Loss: 0.1955 | Acc: 91.83%
Época 2 -> Loss: 0.0779 | Acc: 97.58%
Época 3 -> Loss: 0.0438 | Acc: 98.42%
Época 4 -> Loss: 0.0545 | Acc: 97.92%
Época 5 -> Loss: 0.0246 | Acc: 99.13%


In [18]:
from sklearn.metrics import classification_report, confusion_matrix, recall_score

def evaluate_test(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for signals, labels in test_loader:
            signals, labels = signals.to(device), labels.to(device)
            outputs = model(signals)
            _, predicted = torch.max(outputs, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Métricas clave
    print("--- RESULTADOS EN TEST (BONN) ---")
    print(classification_report(all_labels, all_preds, target_names=['No-Crisis', 'Crisis']))
    
    # Matriz de confusión para ver falsos positivos/negativos
    cm = confusion_matrix(all_labels, all_preds)
    print("Matriz de Confusión:")
    print(cm)
    return all_preds, all_labels

# Asumiendo que guardaste un 'test_loader' aparte (o usa el val_loader si no separaste test)
all_preds, all_labels = evaluate_test(model, test_loader)

--- RESULTADOS EN TEST (BONN) ---
              precision    recall  f1-score   support

   No-Crisis       0.99      0.99      0.99       347
      Crisis       0.99      0.99      0.99       343

    accuracy                           0.99       690
   macro avg       0.99      0.99      0.99       690
weighted avg       0.99      0.99      0.99       690

Matriz de Confusión:
[[343   4]
 [  3 340]]


C:\Users\luissm01\AppData\Local\Temp\ipykernel_11392\156430289.py:9: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(edf_file, preload=True, verbose=False)


NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Ventanas generadas de CHB-MIT: 3612


Crisis detectadas en el archivo: 703


In [24]:
print(y_real)

[0. 0. 0. ... 0. 0. 0.]
